# character identification

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification
from sklearn.model_selection import train_test_split
import torch

file_path = '/content/drive/My Drive/TUGAS AKHIR SATYA/BalineseStoryDataset/charsNamedEntity/train.xlsx'
file_path_test = '/content/drive/My Drive/TUGAS AKHIR SATYA/BalineseStoryDataset/charsNamedEntity/test.xlsx'

# Load annotated dataset
df_train = pd.read_excel(file_path)
df_test = pd.read_excel(file_path_test)
# Only keep relevant columns
df_train = df_train[['StoryID', 'StoryTitle', 'SentenceID', 'Word', 'Character Named Entity Tagset']]
df_test = df_test[['StoryID', 'StoryTitle', 'SentenceID', 'Word', 'Character Named Entity Tagset']]

In [3]:
# Gabungkan untuk dapatkan daftar label unik dari kedua file
combined_df = pd.concat([df_train, df_test], ignore_index=True)

# Buat label2id dan id2label
unique_labels = combined_df['Character Named Entity Tagset'].unique().tolist()
label2id = {label: idx for idx, label in enumerate(unique_labels)}
id2label = {v: k for k, v in label2id.items()}

# Fungsi bantu untuk memproses dataframe jadi list of sentences & label
def prepare_data(df):
    grouped = df.groupby(['StoryID', 'SentenceID'])
    sentences = []
    labels = []
    story_ids = []

    for (story_id, _), group in grouped:
        word_list = group['Word'].tolist()
        label_list = group['Character Named Entity Tagset'].map(label2id).tolist()
        sentences.append(word_list)
        labels.append(label_list)
        story_ids.append(story_id)

    # RETURN sebagai dictionary
    return {
        'tokens': sentences,
        'ner_tags': labels,
        'story_id': story_ids
    }


In [4]:
from datasets import Dataset, DatasetDict

# Proses train dan test
train_data = prepare_data(df_train)
test_data = prepare_data(df_test)

# Ubah ke HuggingFace Dataset
train_dataset = Dataset.from_dict(train_data)
test_dataset = Dataset.from_dict(test_data)

# Gabungkan ke DatasetDict (opsional)
dataset = DatasetDict({
    "train": train_dataset,
    "test": test_dataset
})

model_checkpoint = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

In [5]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        padding='max_length',
        max_length=128
    )

    all_labels = []

    for i, labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                # Token pertama dari suatu kata
                label_ids.append(labels[word_idx])
            else:
                # Subword dari kata yang sama → konversi B-XXX → I-XXX
                original_label = labels[word_idx]
                original_label_name = id2label[original_label]

                if original_label_name.startswith("B-"):
                    i_label_name = "I-" + original_label_name[2:]
                    i_label_id = label2id.get(i_label_name, original_label)
                    label_ids.append(i_label_id)
                else:
                    label_ids.append(original_label)

            previous_word_idx = word_idx

        all_labels.append(label_ids)

    tokenized_inputs["labels"] = all_labels
    return tokenized_inputs

In [6]:
# Apply preprocessing
tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)

# Load model
model = AutoModelForTokenClassification.from_pretrained(model_checkpoint, num_labels=len(label2id), id2label=id2label, label2id=label2id)


Map:   0%|          | 0/4407 [00:00<?, ? examples/s]

Map:   0%|          | 0/2229 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [7]:
pip install seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=ebc4dbd3122de7ee3dc8e53f8fbe3c269a754144b4e951345e6f7ec8f6a6ffa9
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [8]:
from seqeval.metrics import accuracy_score, f1_score, precision_score, recall_score

def compute_metrics(p):
    predictions, labels = p
    predictions = predictions.argmax(axis=-1)

    true_labels = []
    pred_labels = []

    for true, pred in zip(labels, predictions):
        true_seq = []
        pred_seq = []
        for t, p in zip(true, pred):
            if t != -100:
                true_seq.append(id2label[t])
                pred_seq.append(id2label[p])
        true_labels.append(true_seq)
        pred_labels.append(pred_seq)

    return {
        "accuracy": accuracy_score(true_labels, pred_labels),
        "precision": precision_score(true_labels, pred_labels),
        "recall": recall_score(true_labels, pred_labels),
        "f1": f1_score(true_labels, pred_labels),
    }


In [9]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none"
)

data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)


/tmp/ipython-input-3847608069.py:18: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [10]:
!pip uninstall -y wandb

Found existing installation: wandb 0.23.1
Uninstalling wandb-0.23.1:
  Successfully uninstalled wandb-0.23.1


In [11]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.307583,0.906106,0.381518,0.446677,0.411534
2,0.408700,0.220376,0.932141,0.521739,0.672334,0.587540


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.307583,0.906106,0.381518,0.446677,0.411534
2,0.408700,0.220376,0.932141,0.521739,0.672334,0.587540
3,0.408700,0.182230,0.945010,0.632123,0.732998,0.678833
4,0.171000,0.175435,0.945132,0.638339,0.760433,0.694057
5,0.171000,0.166040,0.950308,0.662848,0.782457,0.717703
6,0.110000,0.166057,0.953927,0.696763,0.790185,0.740540
7,0.110000,0.162196,0.956795,0.735046,0.773957,0.754000
8,0.075100,0.182097,0.954224,0.700101,0.806414,0.749506
9,0.075100,0.181215,0.955903,0.719835,0.809119,0.761870
10,0.054300,0.180001,0.957074,0.729863,0.801777,0.764132


TrainOutput(global_step=2760, training_loss=0.15263517939526103, metrics={'train_runtime': 1721.9212, 'train_samples_per_second': 25.594, 'train_steps_per_second': 1.603, 'total_flos': 2879071263843840.0, 'train_loss': 0.15263517939526103, 'epoch': 10.0})

In [12]:
from transformers import AutoModelForTokenClassification

# Contoh: simpan model dan tokenizer (setelah training)
model.save_pretrained("saved_model/my_model")
tokenizer.save_pretrained("saved_model/my_model")


('saved_model/my_model/tokenizer_config.json',
 'saved_model/my_model/special_tokens_map.json',
 'saved_model/my_model/sentencepiece.bpe.model',
 'saved_model/my_model/added_tokens.json',
 'saved_model/my_model/tokenizer.json')

In [13]:
import shutil
from google.colab import files

# Zip
shutil.make_archive("my_model", 'zip', "saved_model/my_model")

# Unduh
files.download("my_model.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
from transformers import TokenClassificationPipeline
from collections import defaultdict
import torch

# Setup pipeline untuk NER
ner_pipe = TokenClassificationPipeline(
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple",  # Gabungkan B- dan I- ke satu entity_group
    device=0 if torch.cuda.is_available() else -1
)

# Bersihkan token dari subword marker
def clean_word(word):
    return word.replace("##", "").strip()

# Simpan hasil karakter berdasarkan story_id
storywise_characters = defaultdict(set)

# Proses setiap kalimat dari dataset test
for story_id, tokens in zip(dataset["test"]["story_id"], dataset["test"]["tokens"]):
    sentence = " ".join(tokens)
    preds = ner_pipe(sentence)

    entity_buffer = []
    prev_entity_type = None

    for pred in preds:
        label = pred["entity_group"]  # misalnya: PNAME, OBJ, ANM
        word = clean_word(pred["word"])

        if label in ["PNAME", "ANM", "GODS", "ADJ", "OBJ"]:  # sesuaikan dengan label karakter
            if entity_buffer:
                name = tokenizer.convert_tokens_to_string(entity_buffer).strip().title()
                if name:
                    storywise_characters[story_id].add(name)
            entity_buffer = [word]
        else:
            if entity_buffer:
                name = tokenizer.convert_tokens_to_string(entity_buffer).strip().title()
                if name:
                    storywise_characters[story_id].add(name)
                entity_buffer = []

        prev_entity_type = label

    # Flush terakhir
    if entity_buffer:
        name = tokenizer.convert_tokens_to_string(entity_buffer).strip().title()
        if name:
            storywise_characters[story_id].add(name)

# ✅ Tampilkan hasil akhir
for story, chars in storywise_characters.items():
    print(f"Story ID: {story}\nCharacters: {sorted(chars)}\n")

Device set to use cuda:0
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Story ID: 0
Characters: ['Bapane', 'Ce', 'Ne', 'Ning', 'Pia', 'Pianakne']

Story ID: 1
Characters: ['Cicing', 'I Angsa', 'I Cicing Gudig', 'I Kerkuak', 'Kerkuak', 'We Angsa']

Story ID: 2
Characters: ['Dedari Supraba', 'Manusa', 'Niwatakawaca', 'Sang Arjuna', 'Sang Hyang Indra', 'Tilottama', 'Widiadarine']

Story ID: 3
Characters: ['Betara', 'Bukit Rangsasa', 'Bupati', 'I Ngurah Rangsasa', 'I Riki', 'Ida Anak Agung Ngurah Rangsasa', 'Ida I Ngurah Sawe', 'Ida Ngurah', 'Ida Sanghyang Sambu', 'Ngurah', 'Ngurahe', 'Paduka Betara', 'Para Bebotoh', 'Pura Rangsasa', 'Rangsasa']

Story ID: 4
Characters: ['Ajinange Guling Ayame Punika', 'Jro Panunggun Kepuh', 'Kepuhe', 'Meonge', 'Pianak Ipun', 'Ulame']

Story ID: 7
Characters: ['Ane', 'Bapa', 'Belibis Putih', 'I Belibis Putih', 'Ida Sang Prabu', 'Ka', 'Kedise', 'Narajana', 'Niki', 'Pandita', 'Panditane', 'Rabi', 'Rabinne', 'Ratu Pandita', 'Sambel']

Story ID: 8
Characters: ['Anake', 'Anake Luh Ento', 'Belog', 'I Belog', 'Kurenane', 'Kurenane I 

In [19]:
import pandas as pd

# Assuming this mapping exists from earlier
story_id_to_title = df_test.groupby('StoryID')['StoryTitle'].first().to_dict()

# Flatten the results into one row per character
char_data_flat = {
    "StoryID": [],
    "judul": [],
    "Character": []
}

for story_id, characters in storywise_characters.items():
    for character in sorted(characters):
        char_data_flat["StoryID"].append(story_id)
        char_data_flat["judul"].append(story_id_to_title.get(story_id, "Unknown"))
        char_data_flat["Character"].append(character)

# Convert to DataFrame
df_flat = pd.DataFrame(char_data_flat)

# Show it
df_flat


,StoryID,judul,Character
0,0,anak_ririh,Bapane
1,0,anak_ririh,Ce
2,0,anak_ririh,Ne
3,0,anak_ririh,Ning
4,0,anak_ririh,Pia
...,...,...,...
1506,164,yuyu_misi_enjekan_kebo_ditundune,I Lutung
1507,164,yuyu_misi_enjekan_kebo_ditundune,I Pekak Dukuh
1508,164,yuyu_misi_enjekan_kebo_ditundune,I Yuyu
1509,164,yuyu_misi_enjekan_kebo_ditundune,Lutung
